# 금융 테크 블로그 → ChromaDB 벡터 저장소 구축
**Runtime → Change runtime type → T4 GPU** 로 설정 후 실행하세요.

In [2]:
# ── 1. 패키지 설치 (로컬/Colab 공용) ──────────────────────────────────────────
# Python 3.13에서는 chromadb가 불안정할 수 있어, 안정 버전으로 고정 설치
!pip install -q "langchain>=0.3,<0.4" "langchain-huggingface>=0.1.2" "langchain-chroma>=0.1.4" "chromadb==0.5.5" "sentence-transformers>=3.0.0" "faiss-cpu>=1.8.0"

In [3]:
# 기존에 꼬였을 수 있는 패키지를 깔끔하게 다시 설치합니다.
!pip install -U langchain langchain-community langchain-core

  Using cached langchain_core-1.3.0-py3-none-any.whl.metadata (4.4 kB)
Using cached langchain_core-1.3.0-py3-none-any.whl (515 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [4]:
# ── 2. GPU 확인 ───────────────────────────────────────────────────────────────
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 모델:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU 사용 가능: True
GPU 모델: NVIDIA GeForce RTX 2060 SUPER
VRAM: 8.2 GB


In [10]:
# ── 3. finance_tech_content.json 업로드 ──────────────────────────────────────
try:
    from google.colab import files
    uploaded = files.upload()
    filename = next(iter(uploaded))
    print("업로드된 파일:", filename)
except ModuleNotFoundError:
    filename = "finance_tech_content.json"  # 로컬 파일 경로
    print("로컬 파일 사용:", filename)


로컬 파일 사용: finance_tech_content.json


In [16]:
import json, re
from datetime import datetime
from urllib.parse import urlparse
from langchain_core.documents import Document
# 1. 랭체인 최신 스플리터 임포트
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 설정값
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 100

# [기존 기능 유지] 텍스트 정제 함수
def clean_text(text):
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    text = re.sub(r'\[[^\]]{1,30}\]', '', text)
    text = re.sub(r'\d{4}[년.\-]\s*\d{1,2}[월.\-]\s*\d{1,2}일?', '', text)
    text = re.sub(r'https?://\S+', '', text)
    for n in ['공유하기','복사하기','카카오톡','트위터','페이스북','링크드인']:
        text = text.replace(n, '')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

# [기존 기능 유지] 날짜 추출 함수
_DATE_PAT = re.compile(r'(\d{4})[.\-년]\s*(\d{1,2})[.\-월]\s*(\d{1,2})')
def extract_date(text):
    m = _DATE_PAT.search(text)
    if m:
        try:
            datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)))
            return f"{m.group(1)}-{m.group(2).zfill(2)}-{m.group(3).zfill(2)}"
        except ValueError:
            pass
    return ''

# [수정된 핵심 함수] 문서 빌드 및 정밀 분할
def build_documents(data):
    docs = []

    # 2. 정밀 절단기(Splitter) 설정
    # 단어 중간이 잘리지 않도록 마침표, 줄바꿈, 공백 등을 기준으로 영리하게 자릅니다.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", " ", ""], # 자르는 우선순위
        is_separator_regex=False,
    )

    for entry in data:
        url, title, content = entry.get('url',''), entry.get('title',''), entry.get('content','')

        # 텍스트 정제
        cleaned  = clean_text(content)

        # 메타데이터 추출 (날짜 및 도메인)
        pub_date = extract_date(cleaned) or extract_date(content)
        try:
            domain = urlparse(url).hostname.removeprefix('www.')
        except:
            domain = ''

        # 3. 랭체인 스플리터를 사용해 텍스트 분할
        chunks = text_splitter.split_text(cleaned)

        for i, chunk in enumerate(chunks):
            # 의미 있는 길이(30자 이상)의 조각만 문서로 만듭니다.
            if len(chunk) > 30:
                docs.append(Document(
                    page_content=chunk,
                    metadata={
                        'source': url,
                        'title': title,
                        'domain': domain,
                        'pub_date': pub_date,
                        'chunk_id': i
                    }
                ))
    return docs

print('✅ 수정된 전처리 함수 정의 완료 (단어 잘림 방지 로직 적용)')

✅ 수정된 전처리 함수 정의 완료 (단어 잘림 방지 로직 적용)


In [19]:
# ── 5. 데이터 로드 및 청킹 ────────────────────────────────────────────────────
from pathlib import Path

candidate_paths = [
    Path('finance_tech_content.json'),
    Path.cwd() / 'finance_tech_content.json',
    Path.cwd() / 'utils' / 'finance_tech_content.json',
]

json_path = next((p for p in candidate_paths if p.exists()), None)
if json_path is None:
    raise FileNotFoundError(
        'finance_tech_content.json 파일을 찾을 수 없습니다. '
        f'현재 작업 경로: {Path.cwd()} / 확인 경로: {candidate_paths}'
    )

with json_path.open(encoding='utf-8') as f:
    data = json.load(f)

docs = build_documents(data)
print(f'사용 파일: {json_path}')
print(f'아티클 {len(data)}개 → 청크 {len(docs)}개')

사용 파일: finance_tech_content.json
아티클 382개 → 청크 6761개


In [20]:
# ── 6. 임베딩 모델 로드 (BAAI/bge-m3, T4 GPU) ────────────────────────────────
from langchain_huggingface import HuggingFaceEmbeddings

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {DEVICE}')

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': DEVICE},
    encode_kwargs={'batch_size': 64, 'normalize_embeddings': True},
)
print('모델 로드 완료')

디바이스: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17336.05it/s]


모델 로드 완료


In [23]:
# ── 7. 벡터스토어 저장 (Chroma 우선, 실패 시 FAISS) ───────────────────────────
from pathlib import Path
from langchain_chroma import Chroma

# Colab이면 /content, 로컬이면 현재 프로젝트 하위에 저장
base_dir = Path('/content') if Path('/content').exists() else Path.cwd()
CHROMA_DIR = str(base_dir / 'chroma_db1')

try:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name='finance_tech',
        persist_directory=CHROMA_DIR,
    )
    print(f'ChromaDB 저장 완료 → {CHROMA_DIR}')
except Exception as exc:
    # chromadb 런타임 이슈(특히 Python 3.13) 시 로컬 파일형 FAISS로 우회
    from langchain_community.vectorstores import FAISS

    faiss_dir = str(base_dir / 'faiss_db1')
    vectorstore = FAISS.from_documents(docs, embeddings)
    vectorstore.save_local(faiss_dir)
    print(f'Chroma 저장 실패: {type(exc).__name__}: {exc}')
    print(f'FAISS 저장 완료 → {faiss_dir}')

ChromaDB 저장 완료 → /home/pj/dev/Tech-Prep-Copilot/utils/chroma_db1


In [32]:
# ── 8. 검색 테스트 ────────────────────────────────────────────────────────────
results = vectorstore.similarity_search('결제 정산 시스템', k=3)
for i, r in enumerate(results, 1):
    print(f'\n[{i}] {r.metadata["title"]} ({r.metadata["domain"]})')
    print(r.page_content[:200])


[1] Spring Batch를 더 우아하게 사용하기 - Spring Batch Plus (d2.naver.com)
네이버페이 정산 시스템은 금액 집계, 지급 요청 등 서로 다른 역할을 하는 수십 개의 Job으로 구성된다. 그런데 이 Job은 상호 의존한다. 이를테면 정산 금액을 집계하지도 않았는데 판매자에게 정산 금액을 지급할 수는 없는 일이다. Spring Batch는 Job 간 의존관계를 설정할 수 있는 JobStep 기능을 제공한다. JobStep을 사용하면 한 

[2] 카카오 AI가 법인카드 영수증을 처리하는 방법: AI 간편 정산 개발기 (tech.kakao.com)
AI 비서의 첫걸음, ‘AI 간편 정산’

'AI 간편 정산하기’는 이름 그대로 AI가 법인카드 정산 과정을 도와주는 기능입니다. 사용자가 여러 장의 영수증 이미지를 한 번에 업로드하면, AI가 각 영수증의 내용을 분석하여 알맞은 법인카드 결제 내역과 자동으로 연결해 줍니다.

AI 간편 정산의 핵심 기능

기능

설명

증빙 서류 자동 매칭

사용자가 모

[3] 카카오 AI가 법인카드 영수증을 처리하는 방법: AI 간편 정산 개발기 (tech.kakao.com)
“직원들이 더 가치 있는 일에 집중하게 할 수는 없을까?”

모든 회사의 고민이자, 기술 조직의 중요한 목표 중 하나일 것입니다. 카카오 사내정보시스템 부서에서는 이 질문에 답하기 위해 AI 기술을 활용하여 직원들의 반복적인 업무를 줄이는 프로젝트를 시작했습니다. 그 첫 번째 결과물이 바로 ‘
AI 간편 정산하기
’ 기능입니다.

수많은 영수증을 하나씩 확


In [34]:
# ── 9. 벡터스토어 폴더(zip) 생성/다운로드 (Colab/로컬 공용) ────────────────────
import shutil
from pathlib import Path

base_dir = Path('/content') if Path('/content').exists() else Path.cwd()

# Chroma 우선, 없으면 FAISS 사용
store_dir = None
for name in ('chroma_db1', 'faiss_db1'):
    candidate = base_dir / name
    if candidate.exists() and candidate.is_dir():
        store_dir = candidate
        break

if store_dir is None:
    raise FileNotFoundError(
        f'압축할 벡터스토어 폴더를 찾을 수 없습니다. 확인 경로: {base_dir}/chroma_db1, {base_dir}/faiss_db1'
    )

zip_base = str(store_dir)
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=str(store_dir))
print(f'압축 완료 → {zip_path}')

# Colab일 때만 브라우저 다운로드 실행
try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
    print(f'다운로드 완료 → {Path(zip_path).name}')
except ModuleNotFoundError:
    print('로컬 환경입니다. 위 zip 경로 파일을 직접 사용하세요.')

압축 완료 → /home/pj/dev/Tech-Prep-Copilot/utils/chroma_db1.zip
로컬 환경입니다. 위 zip 경로 파일을 직접 사용하세요.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ 찾은 데이터 개수: 0개


✅ 데이터 개수: 8695개


In [ ]:
=

Mounted at /content/drive
